In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../dataset/steam_games_messy.csv")

print("Original Dataset Shape:", df.shape)

df.head()

Original Dataset Shape: (500, 8)


,name,release_date,estimated_owners,price,positive,negative,average_playtime_forever,genres
0,Shadow Empire Deluxe 2049,17/05/2013,"0 .. 20,000",5.99,629,163,171,Indie; Horror
1,Crimson Rift Origins,06/12/2013,"100,000 .. 200,000",0.99,6293,23,NaN,Free to Play
2,Forgotten Legends,2012-04-14,"10,000,000 .. 20,000,000",14.99,690,76,152,Strategy
3,Last Empire Origins,2012-03-25,"20,000 .. 50,000",4.99,19009,486,43,Puzzle; Sports
4,Iron Tactics Deluxe 2,2009-05-24,"20,000 .. 50,000",59.99,3820,163,181,Sports; Puzzle


In [2]:
#investigate missing and blank game names.
print("Missing game names:", df["name"].isnull().sum())

print("Blank game names:",
      df["name"].astype("string").str.strip().eq("").sum())

Missing game names: 1
Blank game names: 0


In [3]:
# removing the game name
df["name"] = df["name"].astype("string").str.strip()

df = df[
    df["name"].notna() &
    df["name"].ne("")
].copy()

In [4]:

print("Exact duplicates:", df.duplicated().sum())
df[df.duplicated(keep=False)]

Exact duplicates: 8


,name,release_date,estimated_owners,price,positive,negative,average_playtime_forever,genres
117,Neon Rift Origins 2,08/05/2009,"500,000 .. 1,000,000",39.99,10469,111,356,Adventure; RPG
124,Frozen Dungeon Origins,2014-05-29,unknown,2.99,2190,415,378,Simulation; Casual; Adventure; Massively Multi...
135,Silent Survival Deluxe 2,02/11/2017,"2,000,000 .. 5,000,000",44.99,NaN,89,292,Massively Multiplayer; Adventure
146,Neon Rift Origins 2,08/05/2009,"500,000 .. 1,000,000",39.99,10469,111,356,Adventure; RPG
166,Frozen Rift Reborn,28/06/2025,"50,000 .. 100,000",4.99,20686,58,86,Horror; Simulation
167,Frozen Dungeon Origins,2014-05-29,unknown,2.99,2190,415,378,Simulation; Casual; Adventure; Massively Multi...
202,Silent Arena Origins,May 26 2024,"20,000 .. 50,000",2.69,1684,145,357,Horror; Puzzle; Strategy; Free to Play
231,Beyond Arena Deluxe,April 15 2010,unknown,0.99,2269,949,315,Action; Racing; Massively Multiplayer; Puzzle
259,Silent Survival Deluxe 2,02/11/2017,"2,000,000 .. 5,000,000",44.99,NaN,89,292,Massively Multiplayer; Adventure
359,Forgotten Protocol Origins,April 04 2007,"50,000 .. 100,000",39.99,8469 reviews,397,170,Action


In [5]:
df = df.drop_duplicates().copy()

print("Duplicates after cleaning:", df.duplicated().sum())

Duplicates after cleaning: 0


In [6]:
print(df["price"].unique()[:30])

['5.99' '0.99' '14.99' '4.99' '59.99' '9.99' '19.99' '13.49' '1.79 USD'
 '49.99' ' - ' '17.99' '2.69' '24.99' '$1.99' '$4.99' ' 2.99 ' '1.99'
 '2.99' '44.99' '29.99' '0.0' '0.89' '35.99' '1.79' '8.99' '26.99' '4.49'
 ' 29.99 ' '39.99']


In [7]:
price_text = (
    df["price"]
    .astype("string")
    .str.strip()
    .str.replace(r"(?i)USD", "", regex=True)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df["price"] = pd.to_numeric(price_text, errors="coerce")

In [8]:
print(df["price"].head(20))
print(df["price"].dtype)

0      5.99
1      0.99
2     14.99
3      4.99
4     59.99
5      9.99
6     19.99
7     59.99
8     13.49
9      1.79
10    49.99
11     <NA>
12    17.99
13     0.99
14     2.69
15     2.69
16     <NA>
17     4.99
18    14.99
19    49.99
Name: price, dtype: Float64
Float64


In [9]:
# Convert positive and negative reviews into numeric columns

print(df[["positive", "negative"]].head(20))

def clean_review_count(series):
    return pd.to_numeric(
        series.astype("string")
        .str.replace(",", "", regex=False)
        .str.replace(r"(?i)\s*reviews?\s*$", "", regex=True)
        .str.strip(),
        errors="coerce"
    )

df["positive"] = clean_review_count(df["positive"])
df["negative"] = clean_review_count(df["negative"])

   positive negative
0       629      163
1      6293       23
2       690       76
3     19009      486
4      3820      163
5       289      116
6       785       46
7       923      722
8       NaN      215
9      1025      824
10      355      255
11     6372       68
12      558      189
13     1018      385
14    2029        62
15    32785       77
16     5965       25
17      914       73
18      223      171
19     2627       45


In [10]:
print(df[["positive", "negative"]].dtypes)

print(df[["positive", "negative"]].head(10))

positive    Int64
negative    Int64
dtype: object
   positive  negative
0       629       163
1      6293        23
2       690        76
3     19009       486
4      3820       163
5       289       116
6       785        46
7       923       722
8      <NA>       215
9      1025       824


In [11]:
# Convert playtime into numeric values

print(df["average_playtime_forever"].unique()[:30])

playtime = (
    df["average_playtime_forever"]
    .astype("string")
    .str.strip()
)

# Identify values containing hours or hrs
hours_mask = playtime.str.contains(
    r"(?i)\b(?:hours?|hrs?)\b",
    na=False
)

# Remove commas
playtime = playtime.str.replace(",", "", regex=False)

# Extract numeric values
numeric_playtime = pd.to_numeric(
    playtime.str.extract(
        r"(-?\d+(?:\.\d+)?)",
        expand=False
    ),
    errors="coerce"
)

# Convert hours into minutes
df["average_playtime_forever"] = numeric_playtime.where(
    ~hours_mask,
    numeric_playtime * 60
)

print(df["average_playtime_forever"].head(20))
print(df["average_playtime_forever"].dtype)


['171' nan '152' '43' '181' '85' '38' '172' '87' '232 hrs' '79' '264' '21'
 '944' '240' '114' '166' '101' '165' '601' '424' '250' '174' '1243' '189'
 '1827' '427' '205' '94' '90']
0       171
1      <NA>
2       152
3        43
4       181
5        85
6        38
7       172
8        87
9     13920
10       79
11      264
12     <NA>
13       21
14      944
15      240
16      114
17      166
18      101
19      165
Name: average_playtime_forever, dtype: Int64
Int64


In [12]:
# Standardize release dates
df["release_date"] = pd.to_datetime(
    df["release_date"],
    format="mixed",
    errors="coerce"
)

In [13]:
print(df["release_date"].head(20))
print(df["release_date"].dtype)

0    2013-05-17
1    2013-06-12
2    2012-04-14
3    2012-03-25
4    2009-05-24
5    2007-08-24
6    2018-05-27
7    2011-03-13
8    2013-07-31
9    2020-04-24
10   2011-09-24
11   2014-01-18
12   2013-07-12
13   2014-04-10
14   2007-06-13
15   2012-10-22
16   2024-03-30
17   2020-01-13
18   2019-06-30
19   2011-05-07
Name: release_date, dtype: datetime64[ns]
datetime64[ns]


In [14]:
# Clean genre names
print(df["genres"].unique()[:30])

def clean_genres(value):

    if pd.isna(value):
        return np.nan

    genres = str(value).replace(";", ",")

    genres = [
        genre.strip().title()
        for genre in genres.split(",")
        if genre.strip()
    ]

    # Remove repeated genres while preserving order
    genres = list(dict.fromkeys(genres))

    return ", ".join(genres) if genres else np.nan


df["genres"] = df["genres"].apply(clean_genres)

['Indie; Horror' 'Free to Play' 'Strategy' 'Puzzle; Sports'
 'Sports; Puzzle' 'Puzzle; Simulation; Strategy; Platformer'
 'Massively Multiplayer; Platformer; Racing; Strategy' 'Indie; Sports'
 'STRATEGY' 'Sports' 'Racing' 'Puzzle; Simulation; Adventure; Strategy'
 'Racing; Puzzle' 'Horror' 'Sports; Strategy' 'Casual'
 'Free to Play; Casual' 'Indie'
 'Massively Multiplayer; Action; Free to Play; Horror'
 'Adventure; RPG; Simulation; Strategy' 'Platformer; Racing; RPG' 'Puzzle'
 'Action' 'Casual; Massively Multiplayer; Racing; Adventure'
 'RPG; Indie; Horror; Action' 'Platformer; Strategy'
 'Puzzle; Action; Indie' 'Action; RPG; Sports; Strategy'
 'Free to Play; Adventure; Massively Multiplayer'
 'Indie; Strategy; Platformer']


In [15]:
print(df["genres"].unique()[:30])

['Indie, Horror' 'Free To Play' 'Strategy' 'Puzzle, Sports'
 'Sports, Puzzle' 'Puzzle, Simulation, Strategy, Platformer'
 'Massively Multiplayer, Platformer, Racing, Strategy' 'Indie, Sports'
 'Sports' 'Racing' 'Puzzle, Simulation, Adventure, Strategy'
 'Racing, Puzzle' 'Horror' 'Sports, Strategy' 'Casual'
 'Free To Play, Casual' 'Indie'
 'Massively Multiplayer, Action, Free To Play, Horror'
 'Adventure, Rpg, Simulation, Strategy' 'Platformer, Racing, Rpg' 'Puzzle'
 'Action' 'Casual, Massively Multiplayer, Racing, Adventure'
 'Rpg, Indie, Horror, Action' 'Platformer, Strategy'
 'Puzzle, Action, Indie' 'Action, Rpg, Sports, Strategy'
 'Free To Play, Adventure, Massively Multiplayer'
 'Indie, Strategy, Platformer'
 'Horror, Rpg, Massively Multiplayer, Platformer']


In [16]:
missing_report = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (
        df.isnull().sum() / len(df)
    ) * 100
})

print(missing_report)

                          Missing_Count  Missing_Percentage
name                                  0            0.000000
release_date                         25            5.091650
estimated_owners                      0            0.000000
price                                15            3.054990
positive                             13            2.647658
negative                             12            2.443992
average_playtime_forever             18            3.665988
genres                               13            2.647658


In [17]:
print("========== CLEANED DATASET ==========")

print("\nDataset Shape:")
print(df.shape)

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Records:")
print(df.duplicated().sum())

print("\nFirst Five Records:")
print(df.head())

========== CLEANED DATASET ==========

Dataset Shape:
(491, 8)

Data Types:
name                        string[python]
release_date                datetime64[ns]
estimated_owners                    object
price                              Float64
positive                             Int64
negative                             Int64
average_playtime_forever             Int64
genres                              object
dtype: object

Missing Values:
name                         0
release_date                25
estimated_owners             0
price                       15
positive                    13
negative                    12
average_playtime_forever    18
genres                      13
dtype: int64

Duplicate Records:
0

First Five Records:
                        name release_date          estimated_owners  price  \
0  Shadow Empire Deluxe 2049   2013-05-17               0 .. 20,000   5.99   
1       Crimson Rift Origins   2013-06-12        100,000 .. 200,000   0.99   
2          

In [18]:
print("Negative Prices:", (df["price"] < 0).sum())

print("Negative Positive Reviews:", (df["positive"] < 0).sum())

print("Negative Negative Reviews:", (df["negative"] < 0).sum())

print("Negative Playtime:",
      (df["average_playtime_forever"] < 0).sum())

Negative Prices: 0
Negative Positive Reviews: 0
Negative Negative Reviews: 0
Negative Playtime: 0


In [19]:
df.to_csv(
    "../dataset/steam_games_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!
